# 02b. Оценка прогнозов: метрики, значимость, устойчивость, интервалы

1. **Метрики.** MAE (обязательная), MAPE, WAPE, относительная MAE (`relative_MAE`: MAE модели / MAE naive на тех же точках) и три R²:
   по уровням (как обычно), по изменениям (прогноз прироста к текущему месяцу) и внутри МО (после вычитания
   среднего уровня МО). R² по уровням завышен: он в основном объясняет различия между МО, а не качество прогноза.
2. **Сравнение ансамблей по половинам отсечек.** Кандидаты сравниваются на первой половине отсечек, победитель
   фиксируется и оценивается на второй половине.
3. **Значимость.** Кластерный бутстреп по МО (1000 повторов): 95%-интервал разницы MAE с Prophet.
4. **Разбивки** по классам размера МО и без декабря, лаг публикации таргета (горизонт h+1 вместо h).
5. **Интервалы прогноза** для ансамбля: эмпирические, по ошибкам прошлых отсечек.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = yaml.safe_load(open(ROOT / "configs" / "backtest.yaml", encoding="utf-8"))
RES = ROOT / "results"
rng = np.random.default_rng(CFG["seed"])

P = pd.read_parquet(RES / "predictions_all.parquet")
SAMPLE = set(pd.read_csv(RES / "prophet_sample.csv", dtype={"mo_id": str}).mo_id)
panel = pd.read_parquet(ROOT / CFG["data"]["panel"])
P = P.merge(panel.drop_duplicates("mo_id")[["mo_id", "size_class"]], on="mo_id", how="left")
P["target_date"] = (P.origin.dt.to_period("M") + P.h.astype(int)).dt.to_timestamp()
if "prophet_all" in P:
    P["prophet_sample"] = P.prophet; P["prophet"] = P.prophet_all
else:
    P = P[P.mo_id.isin(SAMPLE)]
P = P.dropna(subset=["y_true"])
print("МО в оценке:", P.mo_id.nunique())

ENS = {"ens_snt_lgbm": ["snt", "lgbm_resid"], "ens_snt_lgbm_news": ["snt", "lgbm_resid_nonat_news"], "ens_sntnat_lgbm": ["snt_nat", "lgbm_resid"],
       "ens_snt_lgbm_logratio": ["snt", "lgbm_logratio"], "ens_prophet_lgbm": ["prophet", "lgbm_resid"],
       "ens_snt_prophet": ["snt", "prophet"]}
for k, cols in ENS.items():
    P[k] = P[cols].mean(axis=1, skipna=False)
MODELS = ["naive", "snt", "snt_nat", "prophet", "prophet_tuned", "lgbm_logratio", "lgbm_resid",
          "lgbm_resid_nonat", "lgbm_resid_nonat_news"] + list(ENS)

def metrics(d, m):
    x = d.dropna(subset=[m, "naive"])
    if x.empty: return {}
    e = x[m] - x.y_true
    mae = e.abs().mean()
    dy, dp = x.y_true - x.naive, x[m] - x.naive
    mo_mean = x.groupby("mo_id").y_true.transform("mean")
    yt_w, yp_w = x.y_true - mo_mean, x[m] - mo_mean
    r2 = lambda a, b: 1 - ((a - b) ** 2).sum() / ((a - a.mean()) ** 2).sum()
    return {"n": len(x), "MAE": mae, "MAPE_%": (e.abs() / x.y_true).mean() * 100,
            "WAPE_%": e.abs().sum() / x.y_true.sum() * 100, "relative_MAE": mae / (x.naive - x.y_true).abs().mean(),
            "R2_уровни": r2(x.y_true, x[m]), "R2_изменения": r2(dy, dp), "R2_внутри_МО": r2(yt_w, yp_w)}

## 1. Метрики по доступным прогнозам
Основные модели оценены на полной панели, усиленный Prophet — на выборке МО. Число наблюдений указано в `n`.

In [ ]:
rows = [{"h": h, "модель": m, **metrics(P[P.h == h], m)} for h in sorted(P.h.unique()) for m in MODELS if m in P]
full = pd.DataFrame([r for r in rows if "MAE" in r]); full.to_csv(RES / "metrics_full.csv", index=False)
main_h = [1, 3, 6, 12]
print("MAE, руб.\n", full[full.h.isin(main_h)].pivot(index="модель", columns="h", values="MAE").round(0).to_string())
print("\nrelative_MAE (меньше 1 — лучше naive)\n", full[full.h.isin(main_h)].pivot(index="модель", columns="h", values="relative_MAE").round(3).to_string())
print("\nR² по изменениям\n", full[full.h.isin(main_h)].pivot(index="модель", columns="h", values="R2_изменения").round(3).to_string())

## 2. Выбор ансамбля на первой половине отсечек, проверка на второй

In [ ]:
CAND = ["snt", "snt_nat", "prophet", "prophet_tuned", "lgbm_resid", "lgbm_logratio", "lgbm_resid_nonat_news"] + list(ENS)
sel = []
for h in [1, 3, 6]:
    d = P[P.h == h]; orig = sorted(d.origin.unique()); half = set(orig[: len(orig) // 2])
    a, b = d[d.origin.isin(half)], d[~d.origin.isin(half)]
    score = {m: metrics(a, m).get("MAE", np.inf) for m in CAND if m in d}
    win = min(score, key=score.get)
    for m in [win, "prophet", "prophet_tuned", "snt"]:
        sel.append({"h": h, "модель": m, "выбрана_на_первой_половине": m == win,
                    "MAE_выбор": score.get(m), "MAE_проверка": metrics(b, m)["MAE"]})
sel = pd.DataFrame(sel).drop_duplicates(["h", "модель"]); sel.to_csv(RES / "ensemble_selection_validated.csv", index=False)
print(sel.round(0).to_string(index=False))

## 3. Значимость: кластерный бутстреп по МО, разница MAE с Prophet

In [ ]:
def boot_diff(d, m, base="prophet", B=1000):
    x = d.dropna(subset=[m, base])
    per = x.assign(a=(x[m] - x.y_true).abs(), b=(x[base] - x.y_true).abs()).groupby("mo_id")[["a", "b"]].agg(["sum", "count"])
    a_s, b_s, cnt = per[("a", "sum")].to_numpy(), per[("b", "sum")].to_numpy(), per[("a", "count")].to_numpy()
    idx = rng.integers(0, len(per), (B, len(per)))
    diff = (a_s[idx].sum(1) - b_s[idx].sum(1)) / cnt[idx].sum(1)
    point = (a_s.sum() - b_s.sum()) / cnt.sum()
    return point, np.percentile(diff, 2.5), np.percentile(diff, 97.5), (diff >= 0).mean()

sig = []
for h in main_h:
    d = P[P.h == h]
    for m in ["snt", "snt_nat", "prophet_tuned", "lgbm_resid", "lgbm_resid_nonat_news", "ens_snt_lgbm", "ens_snt_lgbm_logratio", "ens_snt_lgbm_news"]:
        if m in d and d[m].notna().any():
            pt, lo, hi, p = boot_diff(d, m)
            sig.append({"h": h, "модель": m, "ΔMAE_к_Prophet": pt, "ДИ95_низ": lo, "ДИ95_верх": hi, "p_односторонний": p})
sig = pd.DataFrame(sig); sig.to_csv(RES / "significance_vs_prophet.csv", index=False)
print(sig.round(3).to_string(index=False))

## 4. Разбивки: класс размера МО, без декабря, лаг публикации таргета

In [ ]:
KEY = ["prophet", "snt", "lgbm_resid", "ens_snt_lgbm"]
by_size = pd.DataFrame([{"h": h, "класс": c, "модель": m, **metrics(g, m)} for h in [1, 3, 6]
                        for c, g in P[P.h == h].groupby("size_class") for m in KEY])
by_size.to_csv(RES / "metrics_by_size.csv", index=False)
print(by_size.pivot_table(index=["класс", "модель"], columns="h", values="MAPE_%").round(2).to_string())

nodec = pd.DataFrame([{"h": h, "модель": m, **metrics(P[(P.h == h) & (P.target_date.dt.month != 12)], m)}
                      for h in [1, 3, 6] for m in KEY])
nodec.to_csv(RES / "metrics_without_december.csv", index=False)
print("\nБез декабря, MAE\n", nodec.pivot(index="модель", columns="h", values="MAE").round(0).to_string())

lag = pd.DataFrame([{"h_номинальный": h, "h_с_лагом_1": h + 1, "модель": m,
                     "MAE_без_лага": metrics(P[P.h == h], m).get("MAE"), "MAE_лаг_1": metrics(P[P.h == h + 1], m).get("MAE")}
                    for h in [1, 3, 6] for m in KEY])
lag.to_csv(RES / "metrics_publication_lag.csv", index=False)
print("\nЛаг публикации таргета 1 месяц\n", lag.round(0).to_string(index=False))

## 5. Интервалы прогноза ансамбля (эмпирические)
Для отсечки T берём лог-ошибки ансамбля на всех прошлых прогнозах того же горизонта, чья цель уже наступила
к T, и их 10-й и 90-й перцентили переносим на текущий прогноз. Цель — покрытие около 80%.

In [ ]:
E = "ens_snt_lgbm"
cov = []
for h in [1, 3, 6]:
    d = P[(P.h == h)].dropna(subset=[E]).copy(); d["logerr"] = np.log(d.y_true / d[E])
    for T, g in d.groupby("origin"):
        past = d[d.target_date <= T]
        if len(past) < 300: continue
        lo, hi = np.percentile(past.logerr, [10, 90])
        inside = (g.y_true >= g[E] * np.exp(lo)) & (g.y_true <= g[E] * np.exp(hi))
        cov.append({"h": h, "origin": T, "покрытие": inside.mean(), "ширина_%": (np.exp(hi) - np.exp(lo)) * 100})
cov = pd.DataFrame(cov); cov.to_csv(RES / "interval_coverage.csv", index=False)
print(cov.groupby("h")[["покрытие", "ширина_%"]].mean().round(3).to_string())